# TS2 - Transformada Discreta de Fourier (DFT)

Implementación de un algoritmo que calcula la DFT:

$$X_k = \sum_{n=0}^{N-1} x_n \cdot e^{-j\,2\pi\,k\,n/N}$$

**Firma:**
```
XX = mi_funcion_DFT(xx)

xx: señal a analizar, una matriz (Nx1) de números reales.
XX: DFT de xx, una matriz (Nx1) de números complejos.
```

Validación: la DFT de una senoidal de frecuencia $f_0$ debe ser una delta de Kronecker en $f_0$.

**Bonus:**
- 💎 Comparar con la FFT.
- 👹 DFT de ruido uniforme con $\sigma^2 = 4$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11
})

## Generador de señales (reutilizado de TS1)

In [ ]:
class GeneradorSenales:

    @staticmethod
    def _resolver_parametros(ff, n, fs):
        if fs is None and n is None:
            fs = ff * 2
            n = int(fs)
        elif fs is None:
            fs = n
        elif n is None:
            n = int(fs)
        return int(n), float(fs)

    @staticmethod
    def _generar_tiempo(n, fs):
        t = np.arange(n) / fs
        return t.reshape(-1, 1)

    @staticmethod
    def generar_seno(vmax=1, dc=0, ff=1, ph=0, n=None, fs=None):
        n, fs = GeneradorSenales._resolver_parametros(ff, n, fs)
        tt = GeneradorSenales._generar_tiempo(n, fs)
        xx = dc + vmax * np.sin(2 * np.pi * ff * tt + ph)
        return tt, xx

## Implementación de la DFT

La sumatoria se puede escribir como un producto matricial $X = W \cdot x$, donde $W_{k,n} = e^{-j\,2\pi\,k\,n/N}$ es la matriz DFT (Vandermonde de las raíces N-ésimas de la unidad). Vectorizando con NumPy queda mucho más rápido que dos loops anidados.

In [ ]:
def mi_funcion_DFT(xx):
    """
    Calcula la DFT de una señal real.

    Parametros
    ----------
    xx : array_like (N,) o (N,1)
        Senial real a analizar.

    Retorna
    -------
    XX : ndarray (N,1) complejo
        DFT de xx.
    """
    x = np.asarray(xx, dtype=float).flatten()
    N = x.size

    n = np.arange(N)
    k = n.reshape(-1, 1)

    W = np.exp(-1j * 2 * np.pi * k * n / N)

    XX = W @ x
    return XX.reshape(-1, 1)

## Validación: DFT de una senoidal

Para una senoidal de frecuencia $f_0$ muestreada de modo que $f_0$ caiga exactamente sobre un bin (es decir, $f_0 \cdot N / f_s \in \mathbb{Z}$), el módulo de la DFT debe ser dos deltas de Kronecker: una en $k = f_0\,N/f_s$ y su simétrica en $k = N - f_0\,N/f_s$.

Para $x_n = A\sin(2\pi f_0 n / f_s)$, el módulo de cada delta vale $A\,N/2$.

In [ ]:
N = 1000
fs = 1000
f0 = 10
A = 1.0

tt, xx = GeneradorSenales.generar_seno(vmax=A, ff=f0, n=N, fs=fs)

XX = mi_funcion_DFT(xx)

freqs = np.arange(N) * fs / N
modulo = np.abs(XX).flatten()

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(tt, xx)
axes[0].set_title(f"Senoidal de {f0} Hz")
axes[0].set_xlabel("Tiempo [s]")
axes[0].set_ylabel("Amplitud")

axes[1].stem(freqs[:N//2], modulo[:N//2], basefmt=" ")
axes[1].set_title("|DFT| (mitad positiva del espectro)")
axes[1].set_xlabel("Frecuencia [Hz]")
axes[1].set_ylabel("|X[k]|")
axes[1].set_xlim(0, fs/2)

plt.tight_layout()
plt.show()

k0 = int(f0 * N / fs)
print(f"Pico esperado en k = {k0} (f = {freqs[k0]} Hz)")
print(f"|X[k0]| = {modulo[k0]:.4f}   |   esperado A*N/2 = {A*N/2}")

## 💎 Bonus: Comparación con FFT

La FFT es un algoritmo $O(N \log N)$ que calcula exactamente la misma DFT. Si la implementación es correcta, ambas tienen que coincidir hasta error de máquina.

In [ ]:
XX_mia = mi_funcion_DFT(xx).flatten()
XX_fft = np.fft.fft(xx.flatten())

error_max = np.max(np.abs(XX_mia - XX_fft))
print(f"Error maximo entre mi DFT y np.fft.fft: {error_max:.3e}")

fig, ax = plt.subplots()
ax.plot(freqs[:N//2], np.abs(XX_mia[:N//2]), label="mi_funcion_DFT", linewidth=2)
ax.plot(freqs[:N//2], np.abs(XX_fft[:N//2]), "--", label="np.fft.fft", linewidth=2)
ax.set_xlabel("Frecuencia [Hz]")
ax.set_ylabel("|X[k]|")
ax.set_title("DFT propia vs FFT de NumPy")
ax.set_xlim(0, fs/2)
ax.legend()
plt.tight_layout()
plt.show()

Comparación de velocidad (la diferencia se vuelve enorme cuando $N$ crece):

In [ ]:
import time

for Nb in [256, 1024, 4096]:
    _, x_b = GeneradorSenales.generar_seno(ff=10, n=Nb, fs=Nb)
    x_b = x_b.flatten()

    t0 = time.perf_counter()
    mi_funcion_DFT(x_b)
    t_dft = time.perf_counter() - t0

    t0 = time.perf_counter()
    np.fft.fft(x_b)
    t_fft = time.perf_counter() - t0

    print(f"N = {Nb:5d}  ->  DFT: {t_dft*1e3:8.3f} ms   FFT: {t_fft*1e3:8.3f} ms   ratio: {t_dft/t_fft:.0f}x")

## 👹 Bonus: DFT de ruido uniforme con $\sigma^2 = 4$

Para una distribución uniforme $U(a, b)$ la varianza es:

$$\sigma^2 = \frac{(b - a)^2}{12}$$

Si queremos $\sigma^2 = 4$ y media cero, despejamos $b - a = \sqrt{48} = 4\sqrt{3}$. Tomando $a = -2\sqrt{3}$ y $b = 2\sqrt{3}$ obtenemos ruido blanco uniforme de varianza 4.

El espectro debería ser aproximadamente plano (ruido blanco), con un nivel medio de potencia relacionado con la varianza por el teorema de Parseval:

$$\sum_{n=0}^{N-1} |x_n|^2 = \frac{1}{N}\sum_{k=0}^{N-1} |X_k|^2 \;\Rightarrow\; E[|X_k|^2] \approx N\,\sigma^2$$

In [ ]:
rng = np.random.default_rng(seed=42)

N = 1000
fs = 1000
sigma2 = 4.0

a = -np.sqrt(12 * sigma2) / 2
b =  np.sqrt(12 * sigma2) / 2

ruido = rng.uniform(a, b, size=(N, 1))

print(f"Varianza muestral: {np.var(ruido):.4f}   (esperada: {sigma2})")
print(f"Media muestral:    {np.mean(ruido):.4f}   (esperada: 0)")

XX_ruido = mi_funcion_DFT(ruido)
freqs = np.arange(N) * fs / N

potencia = (np.abs(XX_ruido).flatten() ** 2) / N

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(np.arange(N) / fs, ruido)
axes[0].set_title(f"Ruido uniforme, sigma^2 = {sigma2}")
axes[0].set_xlabel("Tiempo [s]")
axes[0].set_ylabel("Amplitud")

axes[1].plot(freqs[:N//2], potencia[:N//2])
axes[1].axhline(sigma2, color="red", linestyle="--", label=f"sigma^2 = {sigma2}")
axes[1].set_title("|X[k]|^2 / N  (densidad espectral de potencia)")
axes[1].set_xlabel("Frecuencia [Hz]")
axes[1].set_ylabel("Potencia")
axes[1].set_xlim(0, fs/2)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nPotencia media estimada: {np.mean(potencia):.4f}   (esperada ~ {sigma2})")